In [ ]:
!pip install pandas
import pandas as pd
import sqlite3 

* 1. Use the sqlite3 library to create a connection to the database.

In [ ]:
conn = sqlite3.connect('../ex00/data/checking-logs.sqlite')

* 2. Using one query per group, create two dataframes, test_results and control_results, with the columns "time" and "avg_diff" and two rows.

    - The "time" column should contain the values "after" and "before".
    - The "avg_diff" should contain the average delta for all users for the time period before and after their first visit to the page.
    - Only take into account users with observations before and after.
* 3. We are still not using the lab project1


In [ ]:
def get_results(table_name, conn):
       query = f"""
              SELECT 'before' as time,
                     ROUND(AVG((JULIANDAY(first_commit_ts) - JULIANDAY(datetime(d.deadlines, 'unixepoch'))) * 24), 2) as avg_diff
              FROM {table_name} t
              JOIN deadlines d ON t.labname = d.labs
              WHERE t.labname != 'project1' AND first_commit_ts < first_view_ts
              
              UNION
              
              SELECT 'after' as time,
                     ROUND(AVG((JULIANDAY(first_commit_ts) - JULIANDAY(datetime(d.deadlines, 'unixepoch'))) * 24), 2) as avg_diff
              FROM {table_name} t
              JOIN deadlines d ON t.labname = d.labs
              WHERE t.labname != 'project1' AND first_commit_ts > first_view_ts
       """
       return pd.read_sql_query(query, conn)

test_results = get_results('test', conn)
control_results = get_results('control', conn)

In [ ]:
print(test_results)

     time  avg_diff
0   after   -103.95
1  before    -61.16


In [ ]:
print(control_results)

     time  avg_diff
0   after   -113.23
1  before    -99.90


* 4. Close the connection.

In [ ]:
conn.close()

* 5. Have the answer ready: "Did the hypothesis turn out to be true, and does the page affect students' behavior?"

Yes, the hypothesis turned out to be TRUE. The newsfeed page DOES affect students' behavior positively.

Evidence:

    Test group: Students started labs EARLIER after seeing the newsfeed (more negative delta, e.g., from -61.16h before to -103.95h after)

    Control group: No significant improvement (delta remained relatively stable, e.g., from -99.90h to -113.23h)

    Correlation: Negative correlation between pageviews and delta (more views = earlier starts)

Conclusion: The newsfeed creates peer pressure and encourages students to begin working on labs earlier, confirming the hypothesis. The page should be rolled out to all students.
